# Baseline1: add_image_text_source_idx

## Stage1: [all] Split one-to-many mapping and Deal with Obfuscated data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

# 输入数据路径
# data_dir = '../data/hateful_memes'
img_dir = os.path.join(data_dir, 'img')

In [ ]:
print(len(os.listdir(img_dir)))

In [ ]:
splits = ['train', 'dev_seen', 'test_seen']
df = []
for split in splits:
    file_path = os.path.join(data_dir, f'{split}.jsonl')
    split_df = pd.read_json(file_path, lines=True)
    # 数据的类型（train / eval / test?）
    split_df['split'] = split
    # 增添总数据
    # 每一行都添加进来
    df.append(split_df)

df = pd.concat(df, axis=0, ignore_index=True)
df['id'] = df['img'].str.split('/').str[1].str.split('.').str[0]
df.index = df['id']
df.index.name = None
print(df.shape)
df.head()

In [ ]:
# 查看数据分布 (train / eval / test 占数量)
df['split'].value_counts()

In [ ]:
### 这里可以添加可视化 "" ###
### 可做 ###
# 看原数据样本有没有失衡

# plot(piechat(df['label']))

In [ ]:
df['text'].nunique()
# 检查到有重复的文本

In [ ]:
# 去重 (文本去重)
# 为每一句话，添加一个单独的 idx
idx_to_text = {idx:text for idx, text in enumerate(df['text'].unique())}
text_to_idx = {v:k for k, v in idx_to_text.items()}

# 增加新列: 新的idx标识
df['text_idx'] = df['text'].map(text_to_idx)
df.head()

- 出现这种情况，因为论文里面有提到，本数据集有一些 **混淆数据**，用同一段文字加在不同的图片上，就有不同的语义效果，所以元数据中，存有一部分是混淆数据。
- 混淆数据的展现在下面的可视化部分

In [ ]:
# 可视化，看 top-100 重复语句 (发现的确元数据有很多重复的) => 有一句重复高达40+
df['text_idx'].value_counts()[:100].plot.bar(figsize=(30,5))

In [ ]:
# 实现了 "一对多映射" ：即一个文本内容可以对应多个图像
# num_txt idx < num_img
# 找到一个文本下 (单txt_idx) 下，对应的所有图像有这句话图像的 list(id)
# dict => txt_idx : [img_id1, img_id2]
text_idx_to_memes = df.groupby('text_idx')['id'].apply(list).to_dict()

# 字典筛选
text_idx_to_memes = {k:v for k, v in text_idx_to_memes.items() if len(v) > 1}

# 这里是输出: 存在一话多图多情况（output多少句话是有多张图的情况）
print(len(text_idx_to_memes)) # 1497

In [ ]:
# 此时，text_idx_to_memes 是 txt_idx : img_list 的字典组合

In [ ]:
# dict
text_idx_to_label_to_memes = {}

for text_idx, memes in text_idx_to_memes.items():
    # memes: img_list
    # 嵌套字典?
    text_idx_to_label_to_memes[text_idx] = {}
    text_idx_to_label_to_memes[text_idx]['non-hateful'] = []
    text_idx_to_label_to_memes[text_idx]['hateful'] = []

    # 在原来的
    hateful_added, non_hateful_added = False, False
    
    # 在原来上一步已经又的字典组合 dict {txt_idx : img_list} 基础上，再该字典里，再进一步嵌套字典，来记录分别为 "hateful" & "non-hateful" 的数据list
    # (txt_idx, {"" : [ ], "" : [ ]})
    # (2, {'non-hateful': ['13894', '53027'], 'hateful': ['59817', '96180']})
    for meme_idx in memes:
        if df.loc[meme_idx]['label'] == 0:
            text_idx_to_label_to_memes[text_idx]['non-hateful'].append(meme_idx)
            non_hateful_added = True
        else:
            text_idx_to_label_to_memes[text_idx]['hateful'].append(meme_idx)
            hateful_added = True

    if not hateful_added or not non_hateful_added:
        del text_idx_to_label_to_memes[text_idx]

    # if len(text_idx_to_label_to_memes[text_idx]['non-hateful']) == 0:
    #     del text_idx_to_label_to_memes[text_idx]['non-hateful']
    # if len(text_idx_to_label_to_memes[text_idx]['hateful']) == 0:
    #     del text_idx_to_label_to_memes[text_idx]['hateful']

print(len(text_idx_to_label_to_memes))

# 展示效果
list(text_idx_to_label_to_memes.items())[:10]

In [ ]:
# 随机展示一组 “混淆数据”
sample_text_idx = np.random.choice(list(text_idx_to_label_to_memes.keys()))
print(text_idx_to_label_to_memes[sample_text_idx])
meme_idx_1 = np.random.choice(text_idx_to_label_to_memes[sample_text_idx]['non-hateful'])
meme_idx_2 = np.random.choice(text_idx_to_label_to_memes[sample_text_idx]['hateful'])

fig, axes = plt.subplots(1, 2, figsize=(15,10))
axes[0].imshow(Image.open(f'{img_dir}/{meme_idx_1}.png'))
axes[1].imshow(Image.open(f'{img_dir}/{meme_idx_2}.png'))
axes[0].set_title(f'{meme_idx_1}: non-hateful')
axes[1].set_title(f'{meme_idx_2}: hateful')
plt.show()
# end of stage1

---

## Stage2: [txt] Grouping similar sentences

In [ ]:
import spacy
from multiprocessing import Pool

nlp = spacy.load("en_core_web_md")

from sentence_transformers import SentenceTransformer

# 专门用于句子嵌入（Sentence Embedding）任务
model = SentenceTransformer('distilbert-base-nli-mean-tokens')

### Compute Sentence Meaning Similarity
- 这里可以 **后续** 改进 similarity 函数 (本base是基于简单的spcay可计算语义分数, 然后利用余弦相似度)

- 后续改进
> Sentence Transformer	对比学习训练的深层语义嵌入	高	通用、性能最佳，适用于几乎所有语义相似度任务。

> BERT/RoBERTa 分类	句对分类模型	极高（任务特定）	需要大量标注数据进行微调，适用于关系判断任务。

In [ ]:
# 原始版本计算 similarity
def get_simlarity(text1, text2, method='spacy'):
    """
    src: https://stackoverflow.com/questions/65199011/is-there-a-way-to-check-similarity-between-two-full-sentences-in-python
    """
    # 低准确率
    if method == 'jaccard':
        text1 = set(text1.lower().split(" "))
        text2 = set(text2.lower().split(" "))
        score = len(text1.intersection(text2)) / len(text1.union(text2))
    elif method == 'spacy':
        embed1 = nlp(text1)
        embed2 = nlp(text2)
        score = embed1.similarity(embed2)
    else:
        raise ValueError

    return score

In [ ]:
# 按照 txt 划分字典结构，所以已经是去重
meme_idx_to_text = df['text'].to_dict()
meme_idxs = list(meme_idx_to_text.keys())
scores = np.zeros((len(meme_idxs), len(meme_idxs)))

# 全计算 (pair-wise的计算方式)
# 计算复杂度: O^2
# 出来一个matrix
for i in tqdm(range(len(meme_idxs))):
    for j in range(i+1, len(meme_idxs)):
        text_i = meme_idx_to_text[meme_idxs[i]]
        text_j = meme_idx_to_text[meme_idxs[j]]
        score = get_simlarity(text_i, text_j, method='jaccard')
        scores[i, j] = score
        scores[j, i] = score

- Additional: 可补充可视化 (Similarity Score Heatmap / matrix)

In [ ]:
# 补充 Similarity Score Matrix 可视化 （半三角样式）
# insight: 可以观察到 "具有相似性" 的句子多不多，分不多不多 (如果heatmap中，大片都是暖红色，则是有大量相似语义的句子)

In [ ]:
# 为了得出 encoder 默认的 embedding layers 参数值 (# 768)
sentences = list(meme_idx_to_text.values())
sentence_embeddings = model.encode(sentences)
print(sentence_embeddings.shape, type(sentence_embeddings))

In [ ]:
# 计算余弦相似度

from sklearn.metrics.pairwise import cosine_distances, cosine_similarity
scores_tr = cosine_similarity(sentence_embeddings)
for i in range(len(scores_tr)):
    scores_tr[i, i] = 0
print(scores_tr.shape)

# 相似度分数 min / max
print(scores_tr.min(), scores_tr.max())

In [ ]:
# 筛查出相似度分数较高的句子
# 自己设定阈值
# i_idxs / j_idxs 相似度较高的 txt 句子标识 ids
# i_idxs, j_idxs = np.where((scores>0.7) & (scores_tr>0.7))

print(len(i_idxs))
for i, j in zip(i_idxs[:5], j_idxs[:5]):
    print(meme_idxs[i], meme_idxs[j],  f'jaccard: {scores[i, j]}; tr: {scores_tr[i, j]}')
    print(meme_idx_to_text[meme_idxs[i]], '\n', meme_idx_to_text[meme_idxs[j]], '\n')

In [ ]:
meme_idxs = list(meme_idx_to_text.keys())
psuedo_text_idx_to_meme_idxs = {}
for i, j in zip(i_idxs, j_idxs):
    for k in range(len(psuedo_text_idx_to_meme_idxs)):
        if meme_idxs[i] in psuedo_text_idx_to_meme_idxs[k] or meme_idxs[j] in psuedo_text_idx_to_meme_idxs[k]:
            psuedo_text_idx_to_meme_idxs[k].update({meme_idxs[i], meme_idxs[j]})
            break
    else:
        psuedo_text_idx_to_meme_idxs[len(psuedo_text_idx_to_meme_idxs)] = {meme_idxs[i], meme_idxs[j]}
print(len(psuedo_text_idx_to_meme_idxs))

In [ ]:
meme_idx_to_psuedo_text_idx = {}

for psuedo_text_idx, meme_idxs in psuedo_text_idx_to_meme_idxs.items():
    for meme_idx in meme_idxs:
        meme_idx_to_psuedo_text_idx[meme_idx] = psuedo_text_idx
print(len(meme_idx_to_psuedo_text_idx))

# for meme idxs that are not covered
non_covered_meme_idxs = set(df['id'].values) - set(meme_idx_to_psuedo_text_idx.keys())
for i, non_covered_meme_idx in enumerate(non_covered_meme_idxs, start=len(psuedo_text_idx_to_meme_idxs)):
    meme_idx_to_psuedo_text_idx[non_covered_meme_idx] = i
print(len(meme_idx_to_psuedo_text_idx))

In [ ]:
df['pseudo_text_idx'] = df['id'].map(meme_idx_to_psuedo_text_idx)

In [ ]:
df.head()

---

# Stage3: [img] Find Image index

- 这一part是专注于 "img" 的语义处理

In [ ]:
from skimage.metrics import structural_similarity as compare_ssim
import imutils
import cv2

- basemodel: 这里用的是简单的 resnet 来提取特征 (feature_extractor)
    - idea: 这里后续可以 **改进特征的模型**

In [ ]:
import torch
from torchvision.models import resnet18 

model = resnet18(pretrained=True)
model.eval()
feature_extractor = torch.nn.Sequential(*list(model.children())[:-1])
x = torch.randn([1,3,224,224])

output = feature_extractor(x) 
# 只是位置知道 resnet18 的特征嵌入层的 shape
# 无实际意义
print(output.shape)

- basemodel: 这里用的是最简单的图像处理pipeline，简单的resize，tott， norm
    - 后续或许可以改进 img_process pipeline: 提高分辨率？tag? entity? ROIs? ///

In [ ]:
from PIL import Image
import torchvision.transforms as T

# 照片处理
transforms = T.Compose([T.Resize(224), T.ToTensor(), T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

imgs = []

# 全部进行处理
for img_fp in tqdm(df['img'].values.tolist()):
    img = Image.open(f'{data_dir}/{img_fp}').convert('RGB').resize((224, 224))
    img = transforms(img).unsqueeze(dim=0)
    imgs.append(img)

In [ ]:
# 得知处理后的 img_feature .shape
imgs = torch.cat(imgs, dim=0)
print(imgs.shape)

In [ ]:
batch_size = 100
assert len(imgs) % batch_size == 0
num_batches = len(imgs) // batch_size

# 容器
imgs_features = []

# 一个batch为一批次, 提取每个batch中，每张子图的特征 => 拼接
# ：批量（Batch-wise）提取图像特征，并将所有批次的特征拼接
for i in tqdm(range(num_batches)):
    start_idx = i*batch_size
    end_idx = start_idx + batch_size
    imgs_features_batch = feature_extractor(imgs[start_idx:end_idx])
    # 拼接，拉平
    # 转 np
    imgs_features_batch = imgs_features_batch.squeeze().detach().numpy()
    imgs_features.append(imgs_features_batch)
    
    
imgs_features = np.concatenate(imgs_features)
print(imgs_features.shape) # shape: (10000, 512) => (N, D)

In [ ]:
#

- embeddings similarity: tokens / img_embeddings 都可以计算 similarity 分数

In [ ]:
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity

# 
scores_img = cosine_similarity(imgs_features)

for i in range(len(scores_img)):
    scores_img[i, i] = 0
print(scores_img.shape)
print(scores_img.min(), scores_img.max())

In [ ]:
i_idxs, j_idxs = np.where((scores_img>0.92) & (scores_img<1))
print(len(i_idxs))
meme_idx_to_img = df['img'].to_dict()
meme_idxs = list(meme_idx_to_img.keys())
for i, j in zip(i_idxs[:3], j_idxs[:3]):
    print(meme_idxs[i], meme_idxs[j],  f'scores_img: {scores_img[i, j]}')
    fig, axes = plt.subplots(1, 2)
    axes[0].imshow(Image.open(f'{data_dir}/{meme_idx_to_img[meme_idxs[i]]}'))
    axes[1].imshow(Image.open(f'{data_dir}/{meme_idx_to_img[meme_idxs[j]]}'))
    plt.show()

- 一样的操作，和前面的 txt 层面找出 pseudo_txt_idx 一样：
    - 也是找出数据集中和该图片相似度最高的图片的 idx 做参照

In [ ]:
# 照跑即可
# 一样的操作
meme_idxs = list(meme_idx_to_img.keys())
psuedo_img_idx_to_meme_idxs = {}
for i, j in zip(i_idxs, j_idxs):
    for k in range(len(psuedo_img_idx_to_meme_idxs)):
        if meme_idxs[i] in psuedo_img_idx_to_meme_idxs[k] or meme_idxs[j] in psuedo_img_idx_to_meme_idxs[k]:
            psuedo_img_idx_to_meme_idxs[k].update({meme_idxs[i], meme_idxs[j]})
            break
    else:
        psuedo_img_idx_to_meme_idxs[len(psuedo_img_idx_to_meme_idxs)] = {meme_idxs[i], meme_idxs[j]}
print(len(psuedo_img_idx_to_meme_idxs))

meme_idx_to_psuedo_img_idx = {}
for psuedo_img_idx, meme_idxs in psuedo_img_idx_to_meme_idxs.items():
    for meme_idx in meme_idxs:
        meme_idx_to_psuedo_img_idx[meme_idx] = psuedo_img_idx
print(len(meme_idx_to_psuedo_img_idx))

# for meme idxs that are not covered
non_covered_meme_idxs = set(df['id'].values) - set(meme_idx_to_psuedo_img_idx.keys())
for i, non_covered_meme_idx in enumerate(non_covered_meme_idxs, start=len(psuedo_img_idx_to_meme_idxs)):
    meme_idx_to_psuedo_img_idx[non_covered_meme_idx] = i
print(len(meme_idx_to_psuedo_img_idx))

In [ ]:
df['pseudo_img_idx'] = df['id'].map(meme_idx_to_psuedo_img_idx)

In [ ]:
df.head()

# Stage4: Save New CSV Data
- 其实就是添加了4列:
    - split: 数据类型 (train / eval / test)
    - text_idx
    - pseudo_text_idx (定位另一个相似)
    - pseudo_img_idx (定位另一个相似)

In [ ]:
file_path = f'{data_dir}/info.csv'
df.to_csv(file_path, index=False)